### 1. Liiketoiminnan ymmärrys

#### Tavoite
- Tutkia miten uni- ja liikuntatottumukset vaikuttavat hyvinvointiin ja aktiivisuustasoon.

#### Keskeiset kysymykset
- Voiko unen määrällä ja tasolla ennustaa tulevien päivien aktiivisuustasoa taikka hyvinvointia?
- Onko eri käyttäjäryhmillä selkeitä eroja unirytmissä ja liikuntamäärissä?
- Mitkä tekijät vaikuttavat unen määrään ja tasoon?

#### Vaadittava data
- Uni (kesto, taso)
- Liikunta (päivittäiset askeleet, aktiivisuusminuutit, kulutetut kalorit)
- Mahdollisesti lisämuuttujat (syke, stressitaso, unihäiriöt)

#### Mahdolliset rajoitteet
- Datan puute (Unen tasot puuttuvat, joten joudumme käyttämään epäsuoria mittareita kuten sykettä, liikunnan vaikutusta uneen, unen kestoa ja sen säännöllisyyttä.
- Käyttäjien erilaiset elämäntyylit ja taustatekijät
- Älykellodatan mahdollinen epätarkkuus

#### Odotetut tulokset
- Käyttäjäryhmien (klusterien) löytäminen unirytmien ja aktiivisuustottumusten perusteella
- Ennustemalli, joka arvioi aktiivisuustason unen perusteella
- Ymmärrys siitä, miten uni vaikuttaa päivittäiseen hyvinvointiin


### 2. Data understanding: The second phase is to collect and explore the data. What data is available? What are the characteristics of the data (variable types, value distributions etc.)? Are there any quality issues with the data (missing values, outliers, nonsensical values)?

#### Tietoa datasetistä

"Tämä aineisto on kerätty vastaajilta hajautetun kyselyn kautta Amazon Mechanical Turkin avulla ajanjaksolla 03.12.2016–05.12.2016. Kolmekymmentä kelpoisuusehdot täyttävää Fitbit-käyttäjää antoi suostumuksensa henkilökohtaisten aktiivisuusmittaridatojen lähettämiseen, mukaan lukien minuuttitasoiset tiedot fyysisestä aktiivisuudesta, sykkeestä ja unen seurannasta. Eri laitteiden ja käyttäjien yksilölliset seurantatavat ja mieltymykset aiheuttavat vaihtelua aineiston tuloksissa."


Lähde: https://www.kaggle.com/datasets/nurudeenabdulsalaam/fitbit-fitness-tracker-data/data

#### Yleiset muuttujat
1. Id: Käyttäjän yksilöivä tunniste
2. TotalSteps: Päivän aikana otetut askeleet
3. TotalDistance: Kuljettu matka kilometreissä
4. Calories: Arvio kulutetuista kaloreista
5. Sleep.Value: 1 = Sleep / 2 = restless / 3 = Awake

In [ ]:
from operator import index

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.constants import minute

daily_activity = pd.read_csv('archive/dailyActivity_merged.csv')
daily_sleep = pd.read_csv('archive/dailySleep.csv')
weight_data = pd.read_csv('archive/weightLogInfo_merged.csv')
heartrate_data = pd.read_csv('archive/heartrate_seconds_merged.csv')
minuteSleep_data = pd.read_csv('archive/minuteSleep_merged.csv')


# Display the first few rows of each DataFrame to understand the structure
daily_activity.head()

In [ ]:

# Check for missing dates in sleep data
print(daily_sleep['SleepDay'].isnull().sum())

# Check for missing dates in heartrate data
print(heartrate_data['Time'].isnull().sum())

# Check for missing values in weight data
print(weight_data.isnull().sum())

# Check for missing values in daily activity data
print(daily_activity.isnull().sum())

In [ ]:

daily_activity.head()

In [ ]:
heartrate_data.describe()

In [ ]:
daily_activity.describe()

In [ ]:

daily_activity['ActivityDate'] = pd.to_datetime(daily_activity['ActivityDate'])
daily_activity.dtypes

In [ ]:
daily_sleep['SleepDay'] = pd.to_datetime(daily_sleep['SleepDay'])
daily_sleep.dtypes

In [ ]:
weight_data['Date'] = pd.to_datetime(weight_data['Date'])
weight_data.dtypes

### SleepDatan yhdistelyä/ Janin testailua
1. MinuteSleep datasetistä laskettu prosentuaaliet arvot unen tasoille.
    - Ne ryhmitelty päivien ja Id:n mukaan sleepDaily datasettiin.
2. Heartrate ryhmitelty päivän ja Idn mukaan, mutta käytetty suodattimena minuteSleep aikaleimaa, jotta saamme sykkeen vain unen ajalta.
3. Uneen liittyvät taulut yhdistetty uuteen tiedostoon nimeltä dailySleep. Tämä helpottaa tiedon käsittelyä, kun kaikki uneen liittyvä data on yhdessä paikassa

In [ ]:
# Convert the 'Time' column to datetime format
heartrate_data['Time'] = pd.to_datetime(heartrate_data['Time'])

heartrate_data.dtypes

In [ ]:
heartrate_data.head()

In [ ]:
# Extract the date part to group by day
heartrate_data['Date'] = heartrate_data['Time'].dt.date

# Group by 'Id' and 'Date' and aggregate the 'Value' column
daily_heartrate = heartrate_data.groupby(['Id', 'Date']).agg(
    avg_heart_rate=('Value', 'mean'),
    sum_heart_rate=('Value', 'sum'),
    max_heart_rate=('Value', 'max'),
    min_heart_rate=('Value', 'min')
).reset_index()
daily_heartrate.to_csv('archive/dailyHeartrate.csv', index = False)

# Display the first few rows of the grouped data
daily_heartrate.head()

In [ ]:
daily_sleep.head()

In [ ]:
minuteSleep_data['date'] = pd.to_datetime(minuteSleep_data['date'])
minuteSleep_data.dtypes


## SleepPercentages csv luotu.
- Tässä on unen tasot (1,2 ja 3) sen ajalta prosenteissa.

### Yhdistäminen tiedostoihin
- Sleep_stage_percentages nyt yhdistetty sleepDaily tiedostoon ja niistä luotu uusi csv = dailySleep.csv. Alempana kommentoitu code block

In [ ]:
'''
#Convert the 'SleepDay' column to datetime format and extract the date part
daily_sleep['SleepDay'] = pd.to_datetime(daily_sleep['SleepDay'], format='%m/%d/%Y %I:%M:%S %p')
daily_sleep['Date'] = daily_sleep['SleepDay'].dt.date

# Convert the 'Date' column in sleepPercentages to datetime format
sleepPercentages['Date'] = pd.to_datetime(sleepPercentages['Date']).dt.date

# Merge the DataFrames on 'Id' and 'Date'
merged_data = pd.merge(daily_sleep, sleepPercentages, on=['Id', 'Date'], how='left')

# Drop the redundant 'Date' column from the merged DataFrame
merged_data.drop(columns=['Date'], inplace=True)

merged_data.to_csv('archive/dailySleep.csv', index = False)
'''

In [ ]:
# Check if there are missing values in the merge columns
print("Rows with missing values in 'SleepDay':")
print(daily_sleep[daily_sleep['SleepDay'].isna()])

print("\nRows with missing values in 'Time':")
print(heartrate_data[heartrate_data['Time'].isna()])


In [ ]:
print("Data Types in Daily Activity Data:")
print(daily_activity.dtypes, '\n')

print("Data Types in Sleep Data:")
print(daily_sleep.dtypes, '\n')

print("Data Types in Weight Data:")
print(weight_data.dtypes, '\n')

print("Data Types in Heart Rate Data:")
print(heartrate_data.dtypes, '\n')

In [ ]:
print("Summary Statistics for Daily Activity Data:")
print(daily_activity.describe(), '\n')

print("Summary Statistics for Sleep Data:")
print(daily_sleep.describe(), '\n')

print("Summary Statistics for Weight Data:")
print(weight_data.describe(), '\n')

print("Summary Statistics for Heart Rate Data:")
print(heartrate_data.describe(), '\n')

In [ ]:
plt.figure(figsize=(12, 6))

In [ ]:
plt.subplot(2, 2, 1)
sns.histplot(daily_activity['Calories'], kde=True, color='blue')
plt.title('Calories Distribution - Daily Activity')

In [ ]:
plt.subplot(2, 2, 2)
sns.histplot(daily_sleep['TotalMinutesAsleep'], kde=True, color='green')
plt.title('Total Minutes Asleep Distribution - Sleep Data')

In [ ]:
plt.subplot(2, 2, 3)
sns.histplot(weight_data['WeightKg'], kde=True, color='orange')
plt.title('Weight Distribution - Weight Data')

In [ ]:
plt.subplot(2, 2, 4)
sns.histplot(heartrate_data['Value'], kde=True, color='red')
plt.title('Heart Rate Value Distribution - Heart Rate Data')

plt.tight_layout()
plt.show()

In [ ]:

# Step 5: Check for outliers using boxplots
plt.figure(figsize=(12, 6))

# Boxplot for Daily Activity Calories
plt.subplot(2, 2, 1)
sns.boxplot(x=daily_activity['Calories'], color='blue')
plt.title('Calories Boxplot - Daily Activity')

# Boxplot for Sleep TotalMinutesAsleep
plt.subplot(2, 2, 2)
sns.boxplot(x=daily_sleep['TotalMinutesAsleep'], color='green')
plt.title('Total Minutes Asleep Boxplot - Sleep Data')

# Boxplot for Weight
plt.subplot(2, 2, 3)
sns.boxplot(x=weight_data['WeightKg'], color='orange')
plt.title('Weight Boxplot - Weight Data')

# Boxplot for Heart Rate Value
plt.subplot(2, 2, 4)
sns.boxplot(x=heartrate_data['Value'], color='red')
plt.title('Heart Rate Value Boxplot - Heart Rate Data')

plt.tight_layout()
plt.show()

In [ ]:

print("Check for Negative Values in Daily Activity Data:")
print(daily_activity[daily_activity['Calories'] < 0], '\n')

In [ ]:

print("Check for Negative Values in Sleep Data:")
print(daily_sleep[daily_sleep['TotalMinutesAsleep'] < 0], '\n')

In [ ]:

print("Check for Negative or Unrealistic Weight in Weight Data:")
print(weight_data[weight_data['WeightKg'] < 0], '\n')

In [ ]:

print("Check for Unrealistic Heart Rate Values in Heart Rate Data:")
print(heartrate_data[heartrate_data['Value'] < 0], '\n')

### 3. Data preparation: The third phase is to preprocess the data. This includes cleaning the data, transforming the data, and selecting the relevant features. These steps should be documented in such detail that they can be reproduced later.

In [ ]:
## Sydänsyke verrattuna unenlaatuun miinustamalla kokoaika hereilläoloajasta kokoaika nukkumiseen

daily_sleep['TimeDifference'] = daily_sleep['TotalTimeInBed'] - daily_sleep['TotalMinutesAsleep']

# Step 3: Group by 'Id' and calculate the average TotalMinutesAsleep, TotalTimeInBed, and TimeDifference for each Id
sleep_grouped = daily_sleep.groupby('Id').agg(
    avg_total_minutes_asleep=('TotalMinutesAsleep', 'mean'),
    avg_total_time_in_bed=('TotalTimeInBed', 'mean'),
    avg_time_difference_in_bed_vs_asleep=('TimeDifference', 'mean')
).reset_index()

# Step 5: Group by 'Id' and calculate the average, low, and high heart rates for each Id
heartrate_grouped = heartrate_data.groupby('Id')['Value'].agg(['mean', 'min', 'max']).reset_index()

# Step 6: Merge the aggregated sleep data and heart rate data by 'Id'
merged_data = pd.merge(sleep_grouped, heartrate_grouped, on='Id', how='inner')

# Step 7: Rename columns for better clarity
merged_data.rename(columns={
    'mean': 'avg_heart_rate',
    'min': 'min_heart_rate',
    'max': 'max_heart_rate',
}, inplace=True)

# Step 8: Display the first few rows of the renamed merged data
merged_data.head()

In [ ]:

# Compute the correlation between sleep time efficiency (time difference) and heart rates
correlation = merged_data[['avg_time_difference_in_bed_vs_asleep', 'avg_heart_rate', 'min_heart_rate', 'max_heart_rate']].corr()

# Display the correlation matrix
print("Correlation between Sleep Efficiency and Heart Rates:")
print(correlation)


In [ ]:

# Scatter plot between heart rate and TotalMinutesAsleep
plt.figure(figsize=(10, 6))
sns.scatterplot(x=merged_data['Value'], y=merged_data['TotalMinutesAsleep'])
plt.title('Heart Rate vs Total Minutes Asleep')
plt.xlabel('Heart Rate')
plt.ylabel('Total Minutes Asleep')
plt.show()

In [ ]:

# Scatter plot between heart rate and TotalTimeInBed
plt.figure(figsize=(10, 6))
sns.scatterplot(x=merged_data['Value'], y=merged_data['TotalTimeInBed'])
plt.title('Heart Rate vs Total Time in Bed')
plt.xlabel('Heart Rate')
plt.ylabel('Total Time in Bed')
plt.show()

In [ ]:

# Create a simple sleep quality indicator (sleep efficiency)
merged_data['SleepEfficiency'] = merged_data['TotalMinutesAsleep'] / merged_data['TotalTimeInBed']

# Plot Sleep Efficiency against Heart Rate
plt.figure(figsize=(10, 6))
sns.scatterplot(x=merged_data['Value'], y=merged_data['SleepEfficiency'])
plt.title('Heart Rate vs Sleep Efficiency')
plt.xlabel('Heart Rate')
plt.ylabel('Sleep Efficiency')
plt.show()

In [ ]:


from scipy.stats import pearsonr

# Pearson correlation test between Heart Rate and TotalMinutesAsleep
corr_value, p_value = pearsonr(merged_data['Value'], merged_data['TotalMinutesAsleep'])
print(f"Pearson correlation: {corr_value}, p-value: {p_value}")

# Pearson correlation test between Heart Rate and Sleep Efficiency
corr_efficiency, p_value_efficiency = pearsonr(merged_data['Value'], merged_data['SleepEfficiency'])
print(f"Pearson correlation with Sleep Efficiency: {corr_efficiency}, p-value: {p_value_efficiency}")

In [ ]:
from sklearn.preprocessing import StandardScaler

## Ismetin hommelit 
# Assuming your data is loaded into pandas DataFrames
# weight_data: CSV data with weight information
# daily_sleep: CSV data with sleep information

# Merge data based on Id
merged_data = pd.merge(weight_data, daily_sleep, on="Id")

# Standardize the Weight and Sleep data (excluding 'Id' and 'Date')
scaler = StandardScaler()
merged_data[['WeightKg', 'TotalMinutesAsleep', 'TotalTimeInBed']] = scaler.fit_transform(
    merged_data[['WeightKg', 'TotalMinutesAsleep', 'TotalTimeInBed']]
)

# Calculate the difference in time spent in bed and actual sleep
merged_data['TimeDifference'] = merged_data['TotalTimeInBed'] - merged_data['TotalMinutesAsleep']

# You can now perform correlation analysis, such as:
correlation_matrix = merged_data[['WeightKg', 'TotalMinutesAsleep', 'TimeDifference']].corr()

# Output the correlation matrix
print(correlation_matrix)
print('--------------------')
correlation, p_value = pearsonr(merged_data['WeightKg'], merged_data['TimeDifference'])
print(f"Correlation: {correlation}, p-value: {p_value}")


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Assuming 'merged_data' is your DataFrame with the original data
# Select the features you want to cluster
X = merged_data[['WeightKg', 'TotalMinutesAsleep', 'TotalTimeInBed', 'TimeDifference']]

# Step 1: Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Elbow method to determine the optimal number of clusters
inertia = []  # List to store inertia (sum of squared distances)
for k in range(1, 11):  # Try k values from 1 to 10
    model = KMeans(n_clusters=k, random_state=42, init='k-means++', n_init=100)
    model.fit(X_scaled)  # Fit the model using the scaled data
    inertia.append(model.inertia_)  # Store the inertia value for each k

# Plot the elbow method graph
plt.plot(range(1, 11), inertia)
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.show()

# Step 3: Fit KMeans with the optimal number of clusters (based on elbow method)
optimal_k = 4  # You can adjust this value based on the elbow method results
model = KMeans(n_clusters=optimal_k, random_state=42, init='k-means++', n_init=100)
model.fit(X_scaled)  # Fit the model using the scaled data

# Step 4: Add the cluster labels to the DataFrame
merged_data['cluster'] = model.labels_

# Step 5: Get the centroids of the clusters
centroids = model.cluster_centers_

# Step 6: Create the plot for weight and sleep metrics
plt.figure(figsize=(10, 6))

# Scatter plot of the data points, colored by their cluster label
plt.scatter(merged_data['WeightKg'], merged_data['TotalMinutesAsleep'], c=merged_data['cluster'], cmap='viridis', marker='o', label='Data Points')

# Plot the centroids
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, label='Centroids')

plt.xlabel('Weight (Kg)')
plt.ylabel('Total Minutes Asleep')
plt.title('KMeans Clustering of Weight and Sleep Metrics')
plt.legend()
plt.show()

# Step 7: Display the cluster centers (centroids)   

### 4. Modeling: The fourth phase is to choose a machine learning method and train the model. This phase also includes the validation of the model. Documentation needs include: which method was used, which parameters were used, what was the performance of the model?

### 5. Evaluation: The fifth phase is to evaluate the model. How well does the model perform? Does it meet the business requirements?

### 6. Deployment: The final phase is to deploy the model. How will the model be used in practice? How will the results be communicated? This phase may involve creating a recommendation of how to use the model in practice, or what to do next.